# rdepth 보강: 멀티시드(§3.3) + compute-matched(small@400M)

**사용법**: L4 GPU → **모두 실행** → 드라이브 허용 + (처음 한 번) `rdepth_code_v4.zip` 업로드 → 주무세요.
끊겨도 아침에 '모두 실행' 한 번이면 체크포인트에서 이어집니다. 총 ~4.5시간.

In [3]:
!nvidia-smi -L

GPU 0: NVIDIA L4 (UUID: GPU-27f13ac8-4f5c-98db-9562-af4b6721111a)


In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE = '/content/drive/MyDrive/rdepth_out'
os.makedirs(DRIVE, exist_ok=True)
zpath = f'{DRIVE}/rdepth_code_v4.zip'
if not os.path.exists(zpath):
    from google.colab import files
    print('rdepth_code_v4.zip 파일을 선택해 주세요:')
    up = files.upload()
    shutil.move(list(up)[0], zpath)
os.system(f'unzip -q -o {zpath} -d /content/rdepth')
# 중첩 구조 자동 복구 (zip 내부 경로가 rdepth/… 인 경우)
if os.path.exists('/content/rdepth/rdepth/train.py'):
    os.system('cp -r /content/rdepth/rdepth/* /content/rdepth/ && rm -rf /content/rdepth/rdepth')
%cd /content/rdepth
assert os.path.exists('train.py') and '--seed' in open('train.py').read(), '구버전/손상 zip — 드라이브의 rdepth_code_v4.zip 삭제 후 새 zip 재업로드'
print('코드 v4 준비 완료')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/rdepth
코드 v4 준비 완료


In [5]:
%cd /content/rdepth
import os
DRIVE = '/content/drive/MyDrive/rdepth_out'
os.makedirs('data', exist_ok=True)
!cp {DRIVE}/tok4096.json {DRIVE}/val.bin {DRIVE}/train.bin data/
import torch
os.environ['RDEPTH_OUT'] = DRIVE
cap = torch.cuda.get_device_capability()
DTYPE = 'bf16' if cap[0] >= 8 else 'fp16'
print('GPU =', torch.cuda.get_device_name(0), '| dtype =', DTYPE)

/content/rdepth
GPU = NVIDIA L4 | dtype = bf16


In [6]:
# [1/5] sticky, seed 1338
%cd /content/rdepth
!python train.py --run moe-loop-st --max-tokens 100000000 --dtype {DTYPE} --micro-batch 16 --seed 1338 --resume

/content/rdepth
run=moe-loop-st unique_params=41,906,688 resident_fp16=83.8MB dev=cuda dtype=bf16
step 50/3051 loss 6.6729 34,393 tok/s
step 100/3051 loss 5.0825 34,481 tok/s
step 150/3051 loss 4.2393 34,625 tok/s
step 200/3051 loss 3.8821 34,657 tok/s
step 250/3051 loss 3.6338 34,699 tok/s
step 300/3051 loss 3.3105 34,727 tok/s
step 350/3051 loss 3.1515 34,741 tok/s
step 400/3051 loss 2.8976 34,754 tok/s
step 450/3051 loss 2.7785 34,763 tok/s
step 500/3051 loss 2.6208 34,774 tok/s
  [eval] step 500 val 2.6774 uniq_exp 2.000
step 550/3051 loss 2.5405 34,233 tok/s
step 600/3051 loss 2.4053 34,279 tok/s
step 650/3051 loss 2.3212 34,323 tok/s
step 700/3051 loss 2.2136 34,362 tok/s
step 750/3051 loss 2.1904 34,394 tok/s
step 800/3051 loss 2.1877 34,421 tok/s
step 850/3051 loss 2.1360 34,446 tok/s
step 900/3051 loss 2.0656 34,469 tok/s
step 950/3051 loss 2.0081 34,490 tok/s
step 1000/3051 loss 2.1040 34,509 tok/s
  [eval] step 1000 val 2.0953 uniq_exp 2.000
step 1050/3051 loss 2.0167 34,234

In [7]:
# [2/5] re-route, seed 1338
%cd /content/rdepth
!python train.py --run moe-loop-rr --max-tokens 100000000 --dtype {DTYPE} --micro-batch 16 --seed 1338 --resume

/content/rdepth
run=moe-loop-rr unique_params=41,906,688 resident_fp16=83.8MB dev=cuda dtype=bf16
step 50/3051 loss 6.7071 34,274 tok/s
step 100/3051 loss 5.1611 34,520 tok/s
step 150/3051 loss 4.2718 34,551 tok/s
step 200/3051 loss 3.8996 34,571 tok/s
step 250/3051 loss 3.6575 34,595 tok/s
step 300/3051 loss 3.3386 34,605 tok/s
step 350/3051 loss 3.1660 34,611 tok/s
step 400/3051 loss 2.9042 34,615 tok/s
step 450/3051 loss 2.7864 34,617 tok/s
step 500/3051 loss 2.6155 34,621 tok/s
  [eval] step 500 val 2.6771 uniq_exp 2.414
step 550/3051 loss 2.5400 34,086 tok/s
step 600/3051 loss 2.4114 34,131 tok/s
step 650/3051 loss 2.3248 34,169 tok/s
step 700/3051 loss 2.2184 34,203 tok/s
step 750/3051 loss 2.2007 34,234 tok/s
step 800/3051 loss 2.1858 34,262 tok/s
step 850/3051 loss 2.1337 34,287 tok/s
step 900/3051 loss 2.0623 34,308 tok/s
step 950/3051 loss 2.0088 34,324 tok/s
step 1000/3051 loss 2.0960 34,338 tok/s
  [eval] step 1000 val 2.0944 uniq_exp 2.403
step 1050/3051 loss 2.0190 34,072

In [8]:
# [3/5] sticky, seed 1339
%cd /content/rdepth
!python train.py --run moe-loop-st --max-tokens 100000000 --dtype {DTYPE} --micro-batch 16 --seed 1339 --resume

/content/rdepth
run=moe-loop-st unique_params=41,906,688 resident_fp16=83.8MB dev=cuda dtype=bf16
step 50/3051 loss 6.6642 34,519 tok/s
step 100/3051 loss 5.1122 34,709 tok/s
step 150/3051 loss 4.2869 34,757 tok/s
step 200/3051 loss 3.9303 34,776 tok/s
step 250/3051 loss 3.6230 34,795 tok/s
step 300/3051 loss 3.3024 34,808 tok/s
step 350/3051 loss 3.1595 34,816 tok/s
step 400/3051 loss 3.0072 34,822 tok/s
step 450/3051 loss 2.8032 34,824 tok/s
step 500/3051 loss 2.7294 34,825 tok/s
  [eval] step 500 val 2.6862 uniq_exp 2.000
step 550/3051 loss 2.5316 34,277 tok/s
step 600/3051 loss 2.4450 34,323 tok/s
step 650/3051 loss 2.4037 34,365 tok/s
step 700/3051 loss 2.2694 34,400 tok/s
step 750/3051 loss 2.2504 34,431 tok/s
step 800/3051 loss 2.1730 34,457 tok/s
step 850/3051 loss 2.1051 34,481 tok/s
step 900/3051 loss 2.1136 34,501 tok/s
step 950/3051 loss 2.1087 34,521 tok/s
step 1000/3051 loss 2.1140 34,538 tok/s
  [eval] step 1000 val 2.1024 uniq_exp 2.000
step 1050/3051 loss 1.9623 34,269

In [9]:
# [4/5] re-route, seed 1339
%cd /content/rdepth
!python train.py --run moe-loop-rr --max-tokens 100000000 --dtype {DTYPE} --micro-batch 16 --seed 1339 --resume

/content/rdepth
run=moe-loop-rr unique_params=41,906,688 resident_fp16=83.8MB dev=cuda dtype=bf16
step 50/3051 loss 6.6989 34,303 tok/s
step 100/3051 loss 5.1753 34,536 tok/s
step 150/3051 loss 4.3033 34,570 tok/s
step 200/3051 loss 3.9399 34,581 tok/s
step 250/3051 loss 3.6433 34,589 tok/s
step 300/3051 loss 3.3203 34,592 tok/s
step 350/3051 loss 3.1741 34,598 tok/s
step 400/3051 loss 3.0282 34,606 tok/s
step 450/3051 loss 2.8228 34,611 tok/s
step 500/3051 loss 2.7373 34,615 tok/s
  [eval] step 500 val 2.6907 uniq_exp 2.389
step 550/3051 loss 2.5270 34,078 tok/s
step 600/3051 loss 2.4510 34,124 tok/s
step 650/3051 loss 2.4022 34,163 tok/s
step 700/3051 loss 2.2627 34,200 tok/s
step 750/3051 loss 2.2514 34,233 tok/s
step 800/3051 loss 2.1712 34,260 tok/s
step 850/3051 loss 2.1003 34,284 tok/s
step 900/3051 loss 2.1059 34,304 tok/s
step 950/3051 loss 2.0966 34,324 tok/s
step 1000/3051 loss 2.1047 34,342 tok/s
  [eval] step 1000 val 2.0959 uniq_exp 2.380
step 1050/3051 loss 1.9643 34,073

In [10]:
# [5/5] compute-matched: small @ 400M 토큰, from-scratch (시드 1340 = 별도 파일, 스케줄 온전)
%cd /content/rdepth
!python train.py --run small --max-tokens 400000000 --dtype {DTYPE} --micro-batch 64 --seed 1340 --resume

/content/rdepth
run=small unique_params=15,208,960 resident_fp16=30.4MB dev=cuda dtype=bf16
step 50/12207 loss 6.2388 146,970 tok/s
step 100/12207 loss 4.7693 149,536 tok/s
step 150/12207 loss 4.1367 150,308 tok/s
step 200/12207 loss 3.7668 150,690 tok/s
step 250/12207 loss 3.5589 150,935 tok/s
step 300/12207 loss 3.3585 151,134 tok/s
step 350/12207 loss 3.1472 151,292 tok/s
step 400/12207 loss 2.9255 151,426 tok/s
step 450/12207 loss 2.8049 151,535 tok/s
step 500/12207 loss 2.6461 151,610 tok/s
  [eval] step 500 val 2.6805
step 550/12207 loss 2.5224 143,399 tok/s
step 600/12207 loss 2.5059 144,093 tok/s
step 650/12207 loss 2.4283 144,681 tok/s
step 700/12207 loss 2.3561 145,191 tok/s
step 750/12207 loss 2.3097 145,638 tok/s
step 800/12207 loss 2.2280 146,031 tok/s
step 850/12207 loss 2.2136 146,382 tok/s
step 900/12207 loss 2.1225 146,697 tok/s
step 950/12207 loss 2.1018 146,978 tok/s
step 1000/12207 loss 2.0884 147,234 tok/s
  [eval] step 1000 val 2.1394
step 1050/12207 loss 2.0384 1

In [11]:
# 종합: 시드별 sticky/rr + delta 통계, small@400M(from-scratch) vs loop@200M
import csv, os, statistics
out = '/content/drive/MyDrive/rdepth_out'
def best(name):
    p = f'{out}/logs/{name}.csv'
    if not os.path.exists(p): return None
    with open(p) as f: d = list(csv.DictReader(f))
    return min(float(r['val_loss']) for r in d) if d else None
st = {s: best(f'moe-loop-st{"" if s==1337 else f"-s{s}"}') for s in [1337,1338,1339]}
rr = {s: best(f'moe-loop-rr{"" if s==1337 else f"-s{s}"}') for s in [1337,1338,1339]}
print('sticky:', st)
print('re-route:', rr)
ds = [st[s]-rr[s] for s in st if st[s] and rr[s]]
if len(ds) >= 2:
    print(f'delta(st-rr): mean {statistics.mean(ds):+.4f}  std {statistics.stdev(ds):.4f}  n={len(ds)}')
s400 = best('small-s1340')
print(f'small@400M(from-scratch) best: {s400}  vs loop@200M: 1.5577  → compute-matched에서 loop {"승" if s400 and s400>1.5577 else "패/판정불가"}')

sticky: {1337: 1.7129, 1338: 1.7089, 1339: 1.7124}
re-route: {1337: 1.7081, 1338: 1.7098, 1339: 1.7068}
delta(st-rr): mean +0.0032  std 0.0035  n=3
small@400M(from-scratch) best: 1.4936  vs loop@200M: 1.5577  → compute-matched에서 loop 패/판정불가


In [12]:
# [자동 반납] 모든 작업 완료 → GPU 런타임 스스로 해제 (유닛 절약)
# 결과는 전부 드라이브에 저장돼 있으므로 세션이 사라져도 안전
print('모든 런 완료 — 60초 후 런타임을 반납합니다. 결과는 Drive/rdepth_out에 저장됨.')
import time; time.sleep(60)
from google.colab import runtime
runtime.unassign()

모든 런 완료 — 60초 후 런타임을 반납합니다. 결과는 Drive/rdepth_out에 저장됨.
